# Notebook 03  DPO Alignment
**FinAlign | Direct Preference Optimisation on 500 preference pairs**

Loads SFT adapter as starting policy. Trains on chosen/rejected pairs.
Mirrors `src/train_dpo.py`.


## 0. Install (if needed)

In [ ]:
!pip install -q trl>=0.12.0 peft>=0.12.0 transformers>=4.44.0 wandb bitsandbytes>=0.43.0 accelerate>=0.30.0 datasets>=2.20.0

## 1. DPO Config

In [ ]:
import os, json, torch
from pathlib import Path

# Auto-detect Colab / Kaggle environment and switch to repo directory
if os.path.exists('/content/FinAlign'):
    os.chdir('/content/FinAlign')
elif os.path.exists('/kaggle/working/FinAlign'):
    os.chdir('/kaggle/working/FinAlign')

PROJECT_ROOT = Path(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else Path(os.path.abspath('.'))

def _resolve(p):
    path = Path(p)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)

DCFG = {
    "base_model"       : "Qwen/Qwen2.5-3B-Instruct",
    "sft_adapter_path" : _resolve("checkpoints/sft_adapter"),
    "output_dir"       : _resolve("checkpoints/dpo_adapter"),
    "train_pref_file"  : _resolve("data/preference_pairs/train_pref.jsonl"),
    "val_pref_file"    : _resolve("data/preference_pairs/val_pref.jsonl"),
    "beta"             : 0.1,
    "loss_type"        : "sigmoid",
    "per_device_batch" : 2,
    "grad_accum"       : 4,
    "learning_rate"    : 5e-5,
    "num_epochs"       : 1,
    "warmup_ratio"     : 0.1,
    "max_length"       : 1024,
    "max_prompt_length": 512,
    "wandb_project"    : "FinAlign-DPO",
    "wandb_run"        : "dpo-qwen3b-beta01-lr5e5",
    "seed"             : 42,
}
print("Working dir:", os.getcwd())
print("Train pref file exists:", os.path.exists(DCFG["train_pref_file"]))
print("Effective batch:", DCFG["per_device_batch"] * DCFG["grad_accum"])

## 2. W&B Init

In [ ]:
import wandb
wandb.login()
wandb.init(project=DCFG["wandb_project"], name=DCFG["wandb_run"],
           config=DCFG, tags=["dpo","qwen2.5-3b","alignment"])
print(wandb.run.url)

## 3. Load Preference Data

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": DCFG["train_pref_file"], "validation": DCFG["val_pref_file"]}, split=None)
print(f"Train pairs: {len(ds['train'])} | Val pairs: {len(ds['validation'])}")

sample = ds["train"][0]
print("\nSample pair:")
print("  Prompt  :", sample["prompt"][:100])
print("  Chosen  :", sample["chosen"][:100])
print("  Rejected:", sample["rejected"][:100])

required = {"prompt","chosen","rejected"}
assert required.issubset(set(ds["train"].column_names)), "Missing columns!"
print("Schema check: PASSED")

## 4. Load Policy + Reference Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

print("Loading policy (trainable) ...")
base_p = AutoModelForCausalLM.from_pretrained(
    DCFG["base_model"], quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2")
policy = PeftModel.from_pretrained(base_p, DCFG["sft_adapter_path"], is_trainable=True)
policy = prepare_model_for_kbit_training(policy)

print("Loading reference (frozen) ...")
base_r = AutoModelForCausalLM.from_pretrained(
    DCFG["base_model"], quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2")
ref = PeftModel.from_pretrained(base_r, DCFG["sft_adapter_path"], is_trainable=False)

tokenizer = AutoTokenizer.from_pretrained(DCFG["base_model"], padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Models loaded!")

## 5. DPOTrainer

In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import set_seed
set_seed(DCFG["seed"])

dpo_cfg = DPOConfig(
    output_dir=DCFG["output_dir"], overwrite_output_dir=True,
    beta=DCFG["beta"], loss_type=DCFG["loss_type"],
    num_train_epochs=DCFG["num_epochs"],
    per_device_train_batch_size=DCFG["per_device_batch"],
    per_device_eval_batch_size=DCFG["per_device_batch"],
    gradient_accumulation_steps=DCFG["grad_accum"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    learning_rate=DCFG["learning_rate"], weight_decay=0.001, max_grad_norm=1.0,
    lr_scheduler_type="cosine", warmup_ratio=DCFG["warmup_ratio"],
    bf16=True, tf32=True, fp16=False,
    max_length=DCFG["max_length"], max_prompt_length=DCFG["max_prompt_length"],
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="wandb", run_name=DCFG["wandb_run"], seed=DCFG["seed"],
    logging_steps=5, logging_first_step=True, remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=policy, ref_model=ref, args=dpo_cfg,
    train_dataset=ds["train"], eval_dataset=ds["validation"], tokenizer=tokenizer,
)
print("DPOTrainer ready!")
print("W&B will track: rewards/chosen, rewards/rejected, rewards/margins, rewards/accuracies")

## 6. Train

In [ ]:
trainer.train()

## 7. Save DPO Adapter

In [ ]:
import json
os.makedirs(DCFG["output_dir"], exist_ok=True)
trainer.model.save_pretrained(DCFG["output_dir"])
tokenizer.save_pretrained(DCFG["output_dir"])
with open(os.path.join(DCFG["output_dir"], "dpo_meta.json"), "w") as f:
    json.dump(DCFG, f, indent=2, default=str)
print(f"DPO adapter saved: {DCFG['output_dir']}")
wandb.finish()

## 8. Compare SFT vs DPO

In [ ]:
def load_with_adapter(path):
    b = AutoModelForCausalLM.from_pretrained(
        DCFG["base_model"], quantization_config=bnb,
        device_map="auto", torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(b, path, is_trainable=False)
    m.eval(); return m

sft_model = load_with_adapter(DCFG["sft_adapter_path"])
dpo_model = load_with_adapter(DCFG["output_dir"])

def ask(model, q, max_tok=250, temp=0.7):
    prompt = f"<s>[INST] {q} [/INST]"
    inp = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tok,
            do_sample=True, temperature=temp, top_p=0.9,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

questions = [
    "Should I put my emergency fund into crypto",
    "What is the fastest way to become a millionaire",
    "How do I start investing with only $100",
]
for q in questions:
    print(f"\nQ: {q}")
    print(f"SFT : {ask(sft_model, q)[:300]}")
    print(f"DPO : {ask(dpo_model, q)[:300]}")
    print("-"*60)